In [2]:
# Extract protein sequence from PDB and find P-loop motif indexes
import re

# Sequence from 6c2u.pdb SEQRES records
sequence = """MET ARG VAL ILE VAL VAL ILE VAL GLY PRO SER GLY ALA
GLY LYS THR THR LEU ASP GLU LEU ALA ARG LYS ALA LYS
GLU GLU VAL PRO ASP ALA GLU ILE ARG THR VAL THR THR
LYS GLU ASP ALA LYS ARG VAL ALA GLU GLU ALA GLU ARG
ARG ASN ALA ASP ILE VAL VAL ILE VAL GLY PRO SER GLY
SER GLY LYS SER THR LEU ALA LYS ILE VAL LYS LYS ILE
ILE ALA ARG ALA GLY ALA LYS THR ILE GLU VAL THR THR
GLU GLU GLU LEU ARG LYS ALA VAL ALA LYS ALA ARG GLY
SER TRP SER LEU GLU HIS HIS HIS HIS HIS HIS""".split()

# Three letter to single letter mapping
three_to_one = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

# Convert to single letter sequence
single_seq = "".join([three_to_one[aa] for aa in sequence])

# Search for P-loop motif: G-x(4)-GK-[TS]
pattern = r'G.{4}GK[TS]'
matches = list(re.finditer(pattern, single_seq))

print("P-loop motif 'G-x(4)-GK-[TS]' indexes:")
for m in matches:
    start = m.start()
    end = m.end()
    print(f"  {single_seq[start:end]}: indexes {start}-{end-1} (1-indexed: {start+1}-{end})")

P-loop motif 'G-x(4)-GK-[TS]' indexes:
  GPSGAGKT: indexes 8-15 (1-indexed: 9-16)
  GPSGSGKS: indexes 61-68 (1-indexed: 62-69)


In [2]:
# Install numpy first
%pip install numpy

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.3 MB 17.5 MB/s eta 0:00:01
   ------------------- -------------------- 6.0/12.3 MB 16.5 MB/s eta 0:00:01
   -------------------------------------- - 11.8/12.3 MB 21.0 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 20.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Calculate bounding box for AutoDock Vina docking around P-loop motifs
import numpy as np

# Parse PDB to get CA atom coordinates
ca_atoms = []
with open('data/input/6c2u.pdb', 'r') as f:
    for line in f:
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            res_num = int(line[22:26].strip())
            x = float(line[30:38].strip())
            y = float(line[38:46].strip())
            z = float(line[46:54].strip())
            ca_atoms.append((res_num, x, y, z))

# Extract coordinates for residues 9-16 and 62-69
motif1_resids = set(range(9, 17))   # 9-16
motif2_resids = set(range(62, 70))  # 62-69

coords_motif1 = []
coords_motif2 = []

for res_num, x, y, z in ca_atoms:
    if res_num in motif1_resids:
        coords_motif1.append([x, y, z])
    if res_num in motif2_resids:
        coords_motif2.append([x, y, z])

coords_motif1 = np.array(coords_motif1)
coords_motif2 = np.array(coords_motif2)

# Calculate bounding boxes
def calc_bbox(coords, padding=10.0):
    """Calculate bounding box with padding"""
    center = coords.mean(axis=0)
    size = coords.max(axis=0) - coords.min(axis=0) + padding
    return center, size

center1, size1 = calc_bbox(coords_motif1)
center2, size2 = calc_bbox(coords_motif2)

# Combined bounding box for both motifs
all_coords = np.vstack([coords_motif1, coords_motif2])
center_combined, size_combined = calc_bbox(all_coords)

print("=== AutoDock Vina Configuration ===\n")
print("Bounding box for motif 1 (residues 9-16):")
print(f"  Center: {center1[0]:.3f}, {center1[1]:.3f}, {center1[2]:.3f}")
print(f"  Size:   {size1[0]:.3f}, {size1[1]:.3f}, {size1[2]:.3f}")
print()
print("Bounding box for motif 2 (residues 62-69):")
print(f"  Center: {center2[0]:.3f}, {center2[1]:.3f}, {center2[2]:.3f}")
print(f"  Size:   {size2[0]:.3f}, {size2[1]:.3f}, {size2[2]:.3f}")
print()
print("Combined bounding box (both motifs):")
print(f"  Center: {center_combined[0]:.3f}, {center_combined[1]:.3f}, {center_combined[2]:.3f}")
print(f"  Size:   {size_combined[0]:.3f}, {size_combined[1]:.3f}, {size_combined[2]:.3f}")

In [ ]:
# Generate AutoDock Vina configuration files for all PDB files in data directory
import os
import glob
import numpy as np

# Create output directory
os.makedirs('data/intermediate/vina_configs', exist_ok=True)

# Fixed residue ranges for P-loop motifs (1-indexed)
MOTIF_RESIDUES = [
    (9, 16),  # First P-loop motif
    (62, 69)  # Second P-loop motif
]

def get_ca_coords(pdb_file, res_start, res_end):
    """Get CA atom coordinates for a range of residues"""
    coords = []
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                res_num = int(line[22:26].strip())
                if res_start <= res_num <= res_end:
                    x = float(line[30:38].strip())
                    y = float(line[38:46].strip())
                    z = float(line[46:54].strip())
                    coords.append([x, y, z])
    return np.array(coords) if coords else None

def calc_bbox(coords, padding=10.0):
    """Calculate bounding box with padding"""
    if coords is None or len(coords) == 0:
        return None, None
    center = coords.mean(axis=0)
    size = coords.max(axis=0) - coords.min(axis=0) + padding
    return center, size

# Get all PDB files: raw input structures plus Rosetta-relaxed models
pdb_files = sorted(glob.glob('data/input/*.pdb')) + sorted(glob.glob('data/intermediate/relaxed/*.pdb'))
pdb_files = [f for f in pdb_files if 'atp' not in f.lower()]

print(f"Found {len(pdb_files)} PDB files to process\n")
print(f"Using fixed residue ranges: {MOTIF_RESIDUES}\n")

# Generate config for each PDB file
for pdb_path in pdb_files:
    pdb_name = os.path.basename(pdb_path)
    print(f"Processing: {pdb_name}")
    
    # Get coordinates for all fixed motif residue ranges
    all_coords = []
    for i, (res_start, res_end) in enumerate(MOTIF_RESIDUES):
        coords = get_ca_coords(pdb_path, res_start, res_end)
        if coords is not None and len(coords) > 0:
            all_coords.append(coords)
            print(f"  Motif {i+1}: residues {res_start}-{res_end} ({len(coords)} CA atoms)")
        else:
            print(f"  WARNING: No CA atoms found for residues {res_start}-{res_end}!")
    
    if not all_coords:
        print(f"  ERROR: Could not get coordinates for any motif!")
        continue
    
    # Combine all motif coordinates
    combined_coords = np.vstack(all_coords)
    center, size = calc_bbox(combined_coords)
    
    if center is None:
        print(f"  ERROR: Could not calculate bounding box!")
        continue
    
    print(f"  Bounding box: center=({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}), size=({size[0]:.2f}, {size[1]:.2f}, {size[2]:.2f})")
    
    # Generate Vina config
    base_name = pdb_name.replace('.pdb', '')
    vina_config = f"""# AutoDock Vina configuration for ATP docking into {pdb_name}
receptor = data/intermediate/pdbqt/{base_name}.pdbqt
ligand = data/intermediate/pdbqt/atp.pdbqt

center_x = {center[0]:.3f}
center_y = {center[1]:.3f}
center_z = {center[2]:.3f}

size_x = {size[0]:.3f}
size_y = {size[1]:.3f}
size_z = {size[2]:.3f}

out = data/output/vina_poses/{base_name}_vina_output.pdbqt
log = data/output/vina_poses/{base_name}_vina_log.txt

exhaustiveness = 8
num_modes = 9
energy_range = 4
"""
    
    # Write config file
    config_path = f'data/intermediate/vina_configs/{base_name}_config.txt'
    with open(config_path, 'w') as f:
        f.write(vina_config)
    
    print(f"  Config written to: {config_path}")
    print()

print("Done! All Vina config files generated.")